# Notebook A — Convert NIH ChestX-ray14 Dataset into Numbers (Embeddings)

This notebook prepares the full NIH Chest X-ray dataset for training the **Hybrid ConvAE + Potts MRF + Bayesian Anomaly Detection System**.

### High-Efficiency Interleaved Streaming Architecture:
1. **Interleaved Streaming (Self-Cleaning):** Downloads, unzips, and extracts DenseNet-121 embeddings **one archive at a time**, immediately wiping both the zip and extracted images before downloading the next.
2. **Ultra-Low Disk Footprint (~6.5 GB Peak):** The Colab disk never fills up because only 1 archive exists at any time.
3. **Zero RAM / VRAM Spikes:** Memory is purged (`gc.collect()`, `torch.cuda.empty_cache()`) after every archive.
4. **Full Resumption Support:** Every archive part is saved to Google Drive immediately. If interrupted, re-running skips completed archives automatically.
5. **High-Speed Direct CDN Mirror:** Direct HTTPS streaming via Hugging Face (`alkzar90/NIH-Chest-X-ray-dataset`) with zero tokens or 404 errors.


In [ ]:
# Install required dependencies and check hardware
!pip install --upgrade -q opencv-python-headless tqdm requests
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 0. Configuration
Adjust parameters according to your runtime preferences.

In [ ]:
import os, time, math, json, shutil, hashlib, warnings, gc, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION (Interleaved Streaming - Self-Cleaning)
# ==============================================================================
# DATA_SOURCE: 'full' for complete 112,120 image official dataset
DATA_SOURCE = 'full'

# Resolution: Set to 1024 native resolution as requested!
RES         = 1024

# GOOGLE DRIVE STORAGE CONTROL:
# Set to True to save both 1024x1024 full images and embeddings directly to Drive
SAVE_IMAGES = True
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hybrid_pgm_data' + ('_sample' if DATA_SOURCE == 'sample' else '')
os.makedirs(WORK, exist_ok=True)

# Hardware Detection & Tuning for 1024x1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# Safe Batch Size for 1024x1024 (prevents CUDA Out-of-Memory):
if vram_gb >= 35:
    BATCH_SIZE = 16  # 40GB A100
elif vram_gb >= 14:
    BATCH_SIZE = 16  # 15GB T4
else:
    BATCH_SIZE = 8

print(f'GPU Device: {device} ({vram_gb:.1f} GB VRAM) | Batch Size: {BATCH_SIZE}')
print(f'Dataset Mode: {DATA_SOURCE} | Resolution: {RES}x{RES} (Native High-Res)')
print(f'Google Drive Folder: {WORK} (SAVE_IMAGES={SAVE_IMAGES})\n')


## 1. Download Official Metadata and Benchmark Split Lists
Downloads `Data_Entry_2017.csv`, `test_list.txt`, and `BBox_List_2017.csv` directly from the high-speed CDN.

In [ ]:
# ==============================================================================
# 1. DOWNLOAD OFFICIAL METADATA & BENCHMARK SPLIT FILES
# ==============================================================================
root = '/content/nih_metadata'
os.makedirs(root, exist_ok=True)

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None: self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url, dest_path, desc=None):
    if desc is None: desc = os.path.basename(dest_path)
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 100:
        print(f'  ✓ {desc} already exists.')
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=desc) as t:
        urllib.request.urlretrieve(url, filename=dest_path, reporthook=t.update_to)

HF_BASE = 'https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main'

print('Downloading official metadata files...')
meta_csv_path = os.path.join(root, 'Data_Entry_2017.csv')
download_file(f'{HF_BASE}/data/Data_Entry_2017_v2020.csv', meta_csv_path, desc='Data_Entry_2017.csv')

tl_path = os.path.join(root, 'test_list.txt')
download_file(f'{HF_BASE}/data/test_list.txt', tl_path, desc='test_list.txt')

bbox_path = os.path.join(root, 'BBox_List_2017.csv')
download_file(f'{HF_BASE}/data/BBox_List_2017.csv', bbox_path, desc='BBox_List_2017.csv')

# Copy metadata to Google Drive for Notebook B
shutil.copy(meta_csv_path, f'{WORK}/Data_Entry_2017.csv')
shutil.copy(tl_path, f'{WORK}/test_list.txt')
if os.path.exists(bbox_path):
    shutil.copy(bbox_path, f'{WORK}/BBox_List_2017.csv')
print(f'✓ Metadata files saved to Google Drive ({WORK})')


## 2. Interleaved Streaming: 1 Archive at a Time (Self-Cleaning)
Downloads 1 archive ($\sim 3\text{ GB}$), unzips it, extracts DenseNet-121 embeddings, saves to Google Drive, and **immediately deletes the raw images and zip** before starting the next archive.

- **Peak Colab Disk Usage:** $\sim 6.5\text{ GB}$ (instead of $85\text{ GB}$)
- **System RAM:** Stays flat at $\sim 4.2\text{ GB}$ with explicit memory garbage collection.
- **Resumption:** If interrupted, re-running automatically skips archives already saved on Drive.

In [ ]:
# ==============================================================================
# 2. INTERLEAVED STREAMING: DOWNLOAD -> EXTRACT -> DRIVE SAVE -> WIPE DISK -> REPEAT
# ==============================================================================
import cv2, zipfile

# Load Pretrained DenseNet-121 Feature Extractor
print('Loading DenseNet-121 backbone (channels_last, FP16 AMP)...')
try: bb = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)
except Exception: bb = torchvision.models.densenet121(pretrained=True)
bb.classifier = nn.Identity()
bb = bb.to(device).to(memory_format=torch.channels_last).eval()

imn_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imn_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def extract_features(img_array, bs=BATCH_SIZE):
    out = []
    for i in range(0, len(img_array), bs):
        x = torch.from_numpy(img_array[i:i+bs]).to(device).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
        x = ((x - imn_mean) / imn_std).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(bb(x).float().cpu())
    return torch.cat(out).numpy().astype(np.float16)

def load_single_image(path):
    a = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if a is None: return np.zeros((RES, RES), np.uint8)
    a = cv2.resize(a, (RES, RES), interpolation=cv2.INTER_AREA)
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(a)

CACHE_DIR = '/content/nih_stream_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

total_archives = 12 if DATA_SOURCE == 'full' else 1
print(f'Starting Interleaved Streaming across {total_archives} archives...')
print(f'Resolution: {RES}x{RES} | Batch Size: {BATCH_SIZE} | Save Images: {SAVE_IMAGES}\n')

t_start = time.time()

for arch_idx in range(1, total_archives + 1):
    zip_name = f'images_{arch_idx:03d}.zip'
    emb_drive_path = f'{WORK}/emb_archive_{arch_idx:03d}.npy'
    meta_drive_path = f'{WORK}/meta_archive_{arch_idx:03d}.json'
    img_drive_path = f'{WORK}/img_archive_{arch_idx:03d}.npy'

    # Checkpoint check: If already completed, skip!
    need_img = SAVE_IMAGES and not os.path.exists(img_drive_path)
    need_emb = not os.path.exists(emb_drive_path)
    if not (need_img or need_emb):
        print(f'✓ Archive {arch_idx}/{total_archives} ({zip_name}) already on Drive. Skipping.')
        continue

    print('=' * 75)
    print(f'ARCHIVE {arch_idx}/{total_archives}: Processing {zip_name}')
    print('=' * 75)
    t_arch = time.time()

    # 1. Download Zip
    zip_local = os.path.join(CACHE_DIR, zip_name)
    cdn_url = f'{HF_BASE}/data/images/{zip_name}'
    print(f'1. Downloading {zip_name}...')
    download_file(cdn_url, zip_local, desc=zip_name)

    # 2. Extract Zip to temporary folder
    extract_dir = os.path.join(CACHE_DIR, f'extracted_{arch_idx:03d}')
    os.makedirs(extract_dir, exist_ok=True)
    print(f'2. Extracting {zip_name}...')
    with zipfile.ZipFile(zip_local, 'r') as zr:
        zr.extractall(extract_dir)

    # Immediately delete local zip to reclaim local disk space
    os.remove(zip_local)

    # Index all extracted PNGs in this archive
    arch_pngs = sorted([os.path.join(r, f) for r, _, fs in os.walk(extract_dir) for f in fs if f.lower().endswith('.png')])
    arch_files = [os.path.basename(p) for p in arch_pngs]
    n_imgs = len(arch_pngs)
    print(f'   Extracted {n_imgs:,} images.')

    # 3. Process Images & Extract Embeddings (Zero-RAM Memory-Mapped Streaming!)
    CHUNK = min(128, BATCH_SIZE * 4)
    emb_parts = []

    # If SAVE_IMAGES, initialize memory-mapped file on Drive (writes directly to file, 0 MB RAM spike!)
    mmap_img = None
    if SAVE_IMAGES:
        print(f'   Preparing memory-mapped image file: {img_drive_path} (Shape: {n_imgs}x{RES}x{RES})...')
        mmap_img = np.lib.format.open_memmap(
            img_drive_path + '.tmp.npy',
            mode='w+',
            dtype=np.uint8,
            shape=(n_imgs, RES, RES)
        )

    print(f'3. Processing and extracting DenseNet features (micro-chunks of {CHUNK})...')
    for c_start in range(0, n_imgs, CHUNK):
        chunk_paths = arch_pngs[c_start:c_start + CHUNK]
        with ThreadPoolExecutor(8) as ex:
            chunk_imgs = np.stack(list(ex.map(load_single_image, chunk_paths)))
        
        if SAVE_IMAGES and mmap_img is not None:
            mmap_img[c_start:c_start + len(chunk_paths)] = chunk_imgs
        
        emb_parts.append(extract_features(chunk_imgs, bs=BATCH_SIZE))
        del chunk_imgs

    # Finalize memory-mapped image file
    if SAVE_IMAGES and mmap_img is not None:
        mmap_img.flush()
        del mmap_img
        os.replace(img_drive_path + '.tmp.npy', img_drive_path)
        print(f'   ✓ Full images successfully saved to Drive: {img_drive_path}')

    # 4. Save Embeddings directly to Google Drive
    arch_emb = np.concatenate(emb_parts, axis=0)
    np.save(emb_drive_path + '.tmp.npy', arch_emb)
    os.replace(emb_drive_path + '.tmp.npy', emb_drive_path)
    print(f'   ✓ Embeddings successfully saved to Drive: {emb_drive_path}')

    # Save file manifest for this archive
    with open(meta_drive_path, 'w') as f:
        json.dump(arch_files, f)

    # 5. SELF-CLEANING: Wipe the temporary extracted folder from Colab disk!
    shutil.rmtree(extract_dir, ignore_errors=True)

    # Purge RAM & GPU Memory
    del emb_parts, arch_emb, arch_pngs, arch_files
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()

    # Check remaining Colab disk space
    total_disk, used_disk, free_disk = shutil.disk_usage('/content')
    elapsed = time.time() - t_arch
    tot_elapsed = (time.time() - t_start) / 60
    print(f'✓ Archive {arch_idx}/{total_archives} completed! ({elapsed:.1f}s | Total: {tot_elapsed:.1f} min)')
    print(f'  [Self-Cleaning] Raw images deleted | Colab disk free: {free_disk/1e9:.1f} GB\n')

print('🎉 All archives processed and saved to Google Drive successfully!')


## 3. Consolidate Master Embeddings & Build Clean Metadata Table
Combines all 12 archive embeddings into `emb_all.npy`, merges them with `Data_Entry_2017.csv` and the official `test_list.txt` split, and writes `manifest.json`.

In [ ]:
# ==============================================================================
# 3. CONSOLIDATE EMBEDDINGS & BUILD MASTER METADATA (VERIFICATION)
# ==============================================================================
print('Consolidating all archive embeddings and file indices from Drive...')

total_archives = 12 if DATA_SOURCE == 'full' else 1
emb_list = []
all_files = []

for arch_idx in range(1, total_archives + 1):
    ep = f'{WORK}/emb_archive_{arch_idx:03d}.npy'
    mp = f'{WORK}/meta_archive_{arch_idx:03d}.json'
    assert os.path.exists(ep), f'Missing {ep}! Run Section 2 first.'
    assert os.path.exists(mp), f'Missing {mp}! Run Section 2 first.'
    emb_list.append(np.load(ep))
    all_files.extend(json.load(open(mp)))

emb_all = np.concatenate(emb_list, axis=0)
print(f'✓ Consolidated embeddings shape: {emb_all.shape} ({emb_all.nbytes / 1e6:.2f} MB)')
assert len(emb_all) == len(all_files), f'Count mismatch: {len(emb_all)} embeddings vs {len(all_files)} files'
assert np.isfinite(emb_all.astype(np.float32)).all(), 'Non-finite values found in embeddings!'

# Save master embeddings
emb_master_path = f'{WORK}/emb_all.npy'
np.save(emb_master_path, emb_all)
print(f'✓ Master embeddings saved: {emb_master_path}')

# Load and filter Metadata table
print('\nBuilding and formatting master meta.csv...')
raw_meta = pd.read_csv(f'{WORK}/Data_Entry_2017.csv')
raw_meta.columns = [c.strip() for c in raw_meta.columns]
raw_meta = raw_meta.rename(columns={'Image Index': 'file', 'Finding Labels': 'finding', 'Patient ID': 'patient',
                                    'Patient Age': 'age', 'Patient Gender': 'sex', 'View Position': 'view'})
raw_meta = raw_meta[['file', 'finding', 'patient', 'age', 'sex', 'view']]

# Order metadata to match exact ordering of emb_all
file_order_df = pd.DataFrame({'file': all_files, 'idx': np.arange(len(all_files))})
meta = file_order_df.merge(raw_meta, on='file', how='left')
assert len(meta) == len(emb_all), f'Merged metadata length mismatch: {len(meta)} vs {len(emb_all)}'

# Data Cleaning & Disease Flags
meta['age'] = pd.to_numeric(meta.age.astype(str).str.extract(r'(\d+)')[0], errors='coerce')
meta.loc[meta.age > 100, 'age'] = np.nan
meta['age'] = meta.age.fillna(meta.age.median())
meta['is_normal'] = meta.finding.str.strip() == 'No Finding'
meta['fam_alv'] = meta.finding.str.contains('Pneumonia|Infiltration|Consolidation')
meta['fam_eff'] = meta.finding.str.contains('Effusion|Atelectasis')
meta['fam_nod'] = meta.finding.str.contains('Nodule|Mass')

# Official Patient-Level Train/Test Split
tl_file = f'{WORK}/test_list.txt'
if os.path.exists(tl_file):
    tl = set(open(tl_file).read().split())
    meta['split_official'] = np.where(meta.file.isin(tl), 'test', 'trainval')
else:
    rs = np.random.default_rng(0); pats = rs.permutation(meta.patient.unique())
    test_p = set(pats[:int(0.25 * len(pats))])
    meta['split_official'] = np.where(meta.patient.isin(test_p), 'test', 'trainval')

# Save master meta.csv
meta.to_csv(f'{WORK}/meta.csv', index=False)
print(f'✓ Master meta.csv saved: {WORK}/meta.csv ({len(meta):,} records)')
print('  Splits:', meta.split_official.value_counts().to_dict(), '| Abnormal rate:', round(float((~meta.is_normal).mean()), 3))

# Generate Manifest JSON
manifest = {
    'n': int(len(meta)),
    'res': RES,
    'n_archives': int(total_archives),
    'backbone': 'densenet121-imagenet1k-v1 (global-average-pooled, 1024-d, FP16)',
    'emb_dim': int(emb_all.shape[1]),
    'emb_dtype': 'float16',
    'save_images': bool(SAVE_IMAGES),
    'data_source': DATA_SOURCE,
    'clahe': True,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
}
json.dump(manifest, open(f'{WORK}/manifest.json', 'w'), indent=2)
print('✓ Manifest saved:')
print(json.dumps(manifest, indent=2))

# Clean up any temporary stream cache on local Colab
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print('✓ Cleared local temporary cache directory.')

print('\n🎉 NOTEBOOK A COMPLETED SUCCESSFULLY! READY FOR NOTEBOOK B.')


## Done!
Your Google Drive folder (`WORK`) now holds `meta.csv`, `emb_all.npy`, and `manifest.json`.
You can now open **`Hybrid_PGM_B_train_from_drive.ipynb`** to train the models directly from Drive without ever downloading raw images again!